In [1]:
import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import widgets, VBox, Output
from IPython.display import display
from datetime import datetime, timedelta, time, timezone  # ✅ tambah timezone

plt.rcParams['figure.figsize'] = (12, 6)

# ============================
# PARAMETER UTAMA (BISA DITUNING)
# ============================
MIN_SCORE_FOR_TRADE = 1        # minimal |score| biar mau entry (sebelum filter ADX & spike)
MIN_CONFIDENCE = 0.0           # tidak dipakai sebagai hard filter

TP_ATR_MULT = 1.4              # TP = 1.3 * ATR
SL_ATR_MULT = 3.5              # SL = 4.0 * ATR

# Fallback kalau ATR error
FALLBACK_TP_PCT = 0.005        # 0.5%
FALLBACK_SL_PCT = 0.009        # 0.9%

ADX_TREND_THRESHOLD = 20.0     # ADX di atas ini dianggap ada trend
ADX_NO_TRADE_THRESHOLD = 15.0  # ADX di bawah ini: NO TRADE

EXTREME_TR_MULT = 1.6          # jika True Range bar terakhir > 1.3 * ATR → NO TRADE
WICK_RATIO = 2                 # wick > 60% dari range candle → dihindari untuk entry

# Time filter (jam UTC, kira-kira sesi London+NY)
SESSION_START = 13             # 13:00 UTC
SESSION_END = 22               # 22:00 UTC


# ============================
# Download data
# ============================
def fetch_data(ticker_symbol, start_date, end_date, interval):
    stock_data = yf.download(
        ticker_symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False
    )
    return stock_data


# ============================
# ✅ FIX LIVE: buang candle yang sedang berjalan jika end time = jam sekarang (atau berada di candle berjalan)
# ============================
def drop_incomplete_bar_if_live(df: pd.DataFrame, interval: str, end_dt: datetime) -> pd.DataFrame:
    """
    Jika end_dt berada di candle yang sedang berjalan (LIVE), hapus candle berjalan tsb.
    Jika end_dt historis (backtesting / jauh sebelum sekarang), tidak diubah.

    Contoh:
    - sekarang 07:xx UTC, end_dt = 07:00 UTC, interval=1h
      -> candle 07:00 masih berjalan -> dibuang -> terakhir jadi 06:00
    """

    if df is None or df.empty:
        return df

    # Pastikan end_dt UTC aware
    if end_dt.tzinfo is None:
        end_dt = end_dt.replace(tzinfo=timezone.utc)
    else:
        end_dt = end_dt.astimezone(timezone.utc)

    # Pastikan index df UTC
    if isinstance(df.index, pd.DatetimeIndex):
        if df.index.tz is None:
            df = df.copy()
            df.index = df.index.tz_localize("UTC")
        else:
            df = df.tz_convert("UTC")

    now_utc = datetime.now(timezone.utc)

    def floor_to_interval_start(dt: datetime, itv: str) -> datetime:
        """Floor waktu ke awal candle sesuai interval."""
        itv = itv.lower().strip()

        if itv.endswith("h"):
            n = int(itv[:-1])  # misal "1h" -> 1
            base = dt.replace(minute=0, second=0, microsecond=0)
            floored_hour = base.hour - (base.hour % n)
            return base.replace(hour=floored_hour)

        if itv.endswith("m"):
            n = int(itv[:-1])  # misal "15m" -> 15
            base = dt.replace(second=0, microsecond=0)
            floored_min = base.minute - (base.minute % n)
            return base.replace(minute=floored_min)

        if itv.endswith("d"):
            return dt.replace(hour=0, minute=0, second=0, microsecond=0)

        # fallback (kalau interval lain)
        return dt

    current_bar_start = floor_to_interval_start(now_utc, interval)

    # ✅ hanya lakukan trimming kalau memang end_dt berada pada candle berjalan (LIVE)
    # kalau end_dt historis, end_dt < current_bar_start -> tidak diapa-apakan
    if end_dt >= current_bar_start:
        df = df[df.index < current_bar_start]

    return df


# ============================
# EMA, MACD, RSI, ATR, ADX
# ============================
def compute_ema(series, span):
    return series.ewm(span=span, adjust=False).mean()


def compute_macd(series, short_window=12, long_window=26, signal_window=9):
    short_ema = series.ewm(span=short_window, adjust=False).mean()
    long_ema = series.ewm(span=long_window, adjust=False).mean()
    macd = short_ema - long_ema
    signal = macd.rolling(window=signal_window).mean()
    return macd, signal


def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_atr(high, low, close, period=14):
    prev_close = close.shift(1)
    tr1 = (high - low).abs()
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(window=period, min_periods=period).mean()
    return atr


def compute_adx(high, low, close, period=14):
    """
    Perhitungan ADX standar (Wilder's) dengan pandas Series 1D.
    """
    high = high.astype(float)
    low = low.astype(float)
    close = close.astype(float)

    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = up_move.where((up_move > down_move) & (up_move > 0), 0.0)
    minus_dm = down_move.where((down_move > up_move) & (down_move > 0), 0.0)

    prev_close = close.shift(1)
    tr1 = (high - low).abs()
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    tr_smooth = tr.rolling(window=period, min_periods=period).mean()
    plus_dm_smooth = plus_dm.rolling(window=period, min_periods=period).mean()
    minus_dm_smooth = minus_dm.rolling(window=period, min_periods=period).mean()

    plus_di = 100 * (plus_dm_smooth / tr_smooth)
    minus_di = 100 * (minus_dm_smooth / tr_smooth)

    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    adx = dx.rolling(window=period, min_periods=period).mean()

    return adx


# ============================
# Helper: Candle wick filter
# ============================
def is_big_wick_candle(row, wick_ratio=WICK_RATIO):
    """
    row: Series dengan Open, High, Low, Close
    return True kalau upper/lower wick > wick_ratio * range candle
    """
    o = float(row["Open"])
    h = float(row["High"])
    l = float(row["Low"])
    c = float(row["Close"])

    candle_range = h - l
    if candle_range <= 0:
        return False

    upper_wick = h - max(o, c)
    lower_wick = min(o, c) - l

    if upper_wick < 0:
        upper_wick = 0.0
    if lower_wick < 0:
        lower_wick = 0.0

    max_wick = max(upper_wick, lower_wick)
    return max_wick >= wick_ratio * candle_range


# ============================
# Helper: HTF trend (4H EMA50)
# ============================
def compute_htf_trend(ticker, start_date, end_date):
    """
    Hitung bias trend timeframe 4H:
    +1 = uptrend, -1 = downtrend, 0 = netral.
    """
    start_dt = pd.to_datetime(start_date) - timedelta(days=20)
    end_dt = pd.to_datetime(end_date)

    df4 = fetch_data(ticker, start_dt, end_dt, "4h")
    if df4 is None or df4.empty:
        return None

    df4 = df4.dropna()
    if len(df4) < 60:
        return None

    close4 = df4["Close"].astype(float)
    ema50_4h = compute_ema(close4, span=50)
    slope = ema50_4h.diff()

    cond_up = (close4 > ema50_4h) & (slope > 0)
    cond_dn = (close4 < ema50_4h) & (slope < 0)

    trend = pd.Series(0, index=df4.index, dtype=float)
    trend.loc[cond_up[cond_up].index] = 1
    trend.loc[cond_dn[cond_dn].index] = -1

    return trend


# ============================
# Hitung signal & confidence
# ============================
def compute_signal_from_indicators(df):
    """
    df harus punya kolom: Open, High, Low, Close
    Return: dict dengan posisi, TP/SL, confidence, dan indikator utama
    """
    open_ = df["Open"].astype(float)
    close = df["Close"].astype(float)
    high = df["High"].astype(float)
    low = df["Low"].astype(float)

    # Indikator utama
    ema_fast = compute_ema(close, span=21)
    ema_slow = compute_ema(close, span=50)
    macd, signal = compute_macd(close)
    rsi = compute_rsi(close, period=14)
    atr = compute_atr(high, low, close, period=14)
    adx = compute_adx(high, low, close, period=14)

    # Ambil nilai terakhir
    last_close = float(close.iloc[-1])
    ema_fast_last = float(ema_fast.iloc[-1])
    ema_slow_last = float(ema_slow.iloc[-1])
    macd_last = float(macd.dropna().iloc[-1]) if macd.dropna().size > 0 else np.nan
    signal_last = float(signal.dropna().iloc[-1]) if signal.dropna().size > 0 else np.nan
    rsi_last = float(rsi.dropna().iloc[-1]) if rsi.dropna().size > 0 else np.nan
    atr_last = float(atr.dropna().iloc[-1]) if atr.dropna().size > 0 else np.nan
    adx_last = float(adx.dropna().iloc[-1]) if adx.dropna().size > 0 else np.nan

    # ============================
    # Skoring bullish vs bearish
    # ============================
    score = 0
    max_score = 6  # 6 faktor dalam scoring

    # 1) Price vs EMA fast
    if last_close > ema_fast_last:
        score += 1
    else:
        score -= 1

    # 2) EMA fast vs EMA slow (trend)
    if ema_fast_last > ema_slow_last:
        score += 1
    else:
        score -= 1

    # 3) MACD vs Signal (momentum)
    if not np.isnan(macd_last) and not np.isnan(signal_last):
        if macd_last > signal_last:
            score += 1
        else:
            score -= 1

    # 4) MACD di atas / bawah nol
    if not np.isnan(macd_last):
        if macd_last > 0:
            score += 1
        else:
            score -= 1

    # 5) RSI (overbought/oversold soft)
    if not np.isnan(rsi_last):
        if rsi_last > 55:
            score += 1
        elif rsi_last < 45:
            score -= 1
        # 45–55 = netral

    # 6) ADX (kekuatan trend)
    if not np.isnan(adx_last) and adx_last >= ADX_TREND_THRESHOLD:
        if ema_fast_last > ema_slow_last:
            score += 1  # trend naik, ADX tinggi → bullish point
        elif ema_fast_last < ema_slow_last:
            score -= 1  # trend turun, ADX tinggi → bearish point

    # ============================
    # Posisi awal (sebelum filter)
    # ============================
    if score >= MIN_SCORE_FOR_TRADE:
        raw_position = "LONG"
    elif score <= -MIN_SCORE_FOR_TRADE:
        raw_position = "SHORT"
    else:
        raw_position = "NO TRADE"

    # ============================
    # TP & SL dari ATR (fixed multiple)
    # ============================
    if np.isnan(atr_last) or atr_last <= 0:
        atr_based = False
        tp_pct = FALLBACK_TP_PCT
        sl_pct = FALLBACK_SL_PCT
    else:
        atr_based = True
        atr_pct = atr_last / last_close
        tp_pct = TP_ATR_MULT * atr_pct
        sl_pct = SL_ATR_MULT * atr_pct

    # Hitung TP/SL price (saran)
    if raw_position == "LONG":
        tp_price = last_close * (1 + tp_pct)
        sl_price = last_close * (1 - sl_pct)
    elif raw_position == "SHORT":
        tp_price = last_close * (1 - tp_pct)
        sl_price = last_close * (1 + sl_pct)
    else:
        tp_price = np.nan
        sl_price = np.nan

    # ============================
    # Confidence dari |score|
    # ============================
    strength = min(abs(score) / max_score, 1.0)
    confidence = 50 + strength * 40  # 50–90%

    # ============================
    # Label kualitas sinyal
    # ============================
    if confidence >= 75 and not np.isnan(adx_last) and adx_last >= 20:
        quality = "STRONG"
    elif confidence >= 60:
        quality = "MEDIUM"
    else:
        quality = "WEAK"

    # ============================
    # HARD FILTER 1: ADX terlalu rendah → NO TRADE
    # ============================
    filter_reason = []
    position_after_filters = raw_position

    if not np.isnan(adx_last) and adx_last < ADX_NO_TRADE_THRESHOLD:
        position_after_filters = "NO TRADE"
        filter_reason.append(f"ADX < {ADX_NO_TRADE_THRESHOLD}")

    # ============================
    # HARD FILTER 2: Spike/TR ekstrem di candle terakhir → NO TRADE
    # ============================
    prev_close = float(close.iloc[-2]) if len(close) > 1 else float(last_close)
    tr1 = float(abs(high.iloc[-1] - low.iloc[-1]))
    tr2 = float(abs(high.iloc[-1] - prev_close))
    tr3 = float(abs(low.iloc[-1] - prev_close))
    last_tr = max(tr1, tr2, tr3)

    if not np.isnan(atr_last) and atr_last > 0 and last_tr > EXTREME_TR_MULT * atr_last:
        position_after_filters = "NO TRADE"
        filter_reason.append(f"Last TR > {EXTREME_TR_MULT} * ATR (spike)")

    result = {
        "last_close": last_close,
        "ema_fast_last": ema_fast_last,
        "ema_slow_last": ema_slow_last,
        "macd_last": macd_last,
        "signal_last": signal_last,
        "rsi_last": rsi_last,
        "atr_last": atr_last,
        "adx_last": adx_last,
        "score": score,
        "raw_position": raw_position,
        "position": position_after_filters,
        "tp_pct": tp_pct * 100,
        "sl_pct": sl_pct * 100,
        "tp_price": tp_price,
        "sl_price": sl_price,
        "confidence_pct": confidence,
        "quality_label": quality,
        "atr_based": atr_based,
        "last_tr": float(last_tr),
        "filter_reason": "; ".join(filter_reason) if filter_reason else "-"
    }

    return result, ema_fast, ema_slow, macd, signal, rsi, adx


# ====================================================
# BACKTEST SEDERHANA (FUTURES, LONG/SHORT, TP/SL)
# ====================================================
def backtest_strategy(
    ticker,
    start_date,
    end_date,
    interval="1h",
    min_bars=80,
    initial_equity=1.0,
    verbose=False
):
    """
    Backtest forward-walk:
    - Di tiap bar, hitung signal pakai data sampai bar itu saja (no lookahead).
    - Entry di close ketika signal muncul dan tidak sedang ada posisi.
    - Exit kalau harga close menyentuh TP/SL (approx).
    - Tambahan:
        * Time filter (jam UTC)
        * Wick filter
        * HTF 4H EMA50 trend filter
        * Break-even rule hanya di trend kuat
    """
    print(f"Download data {ticker} dari {start_date} sampai {end_date} ({interval}) ...")
    df = fetch_data(ticker, start_date, end_date, interval)
    df = df.dropna()

    if len(df) < min_bars:
        print(f"⚠️ Data terlalu sedikit (len={len(df)}, butuh minimal {min_bars}).")
        return None, None

    close = df["Close"].astype(float)

    # Hitung HTF 4H trend (bias)
    htf_trend = compute_htf_trend(ticker, start_date, end_date)
    if htf_trend is not None:
        htf_on_ltf = htf_trend.reindex(df.index, method="ffill").fillna(0)
    else:
        htf_on_ltf = pd.Series(0, index=df.index)

    in_position = False
    pos_side = None  # 1 = long, -1 = short
    entry_price = None
    tp_price = None
    sl_price = None
    entry_time = None
    moved_to_be = False
    be_allowed = False
    be_full_tp_pct = None
    adx_entry = None

    trades = []
    equity = [initial_equity]
    equity_index = [df.index[0]]

    for i in range(min_bars, len(df)):
        current_time = df.index[i]
        current_close = float(close.iloc[i])

        # ============================
        # Kalau sedang ada posisi → cek BE rule + TP/SL
        # ============================
        if in_position:
            ret = 0.0
            exit_reason = None

            # BREAK-EVEN RULE (hanya kalau diizinkan)
            if be_allowed and not moved_to_be and be_full_tp_pct is not None and be_full_tp_pct > 0:
                if pos_side == 1:  # LONG
                    curr_pct = (current_close - entry_price) / entry_price
                else:              # SHORT
                    curr_pct = (entry_price - current_close) / entry_price

                # baru geser SL ke BE kalau sudah jalan >= 50% menuju TP
                if curr_pct >= 0.5 * be_full_tp_pct:
                    sl_price = entry_price
                    moved_to_be = True

            # Cek TP/SL dengan harga close (simple)
            if pos_side == 1:  # LONG
                if current_close >= tp_price:
                    ret = (tp_price - entry_price) / entry_price
                    exit_reason = "TP"
                elif current_close <= sl_price:
                    ret = (sl_price - entry_price) / entry_price
                    exit_reason = "SL"
            else:  # SHORT
                if current_close <= tp_price:
                    ret = (entry_price - tp_price) / entry_price
                    exit_reason = "TP"
                elif current_close >= sl_price:
                    ret = (entry_price - sl_price) / entry_price
                    exit_reason = "SL"

            if exit_reason is not None:
                new_equity = equity[-1] * (1.0 + ret)
                equity.append(new_equity)
                equity_index.append(current_time)

                trades.append({
                    "entry_time": entry_time,
                    "exit_time": current_time,
                    "side": "LONG" if pos_side == 1 else "SHORT",
                    "entry_price": entry_price,
                    "exit_price": current_close,
                    "tp_price": tp_price,
                    "sl_price": sl_price,
                    "ret_pct": ret * 100.0,
                    "exit_reason": exit_reason,
                    "adx_entry": adx_entry
                })

                in_position = False
                pos_side = None
                entry_price = None
                tp_price = None
                sl_price = None
                entry_time = None
                moved_to_be = False
                be_allowed = False
                be_full_tp_pct = None
                adx_entry = None

                continue  # jangan entry lagi di bar yang sama

        # ============================
        # Kalau tidak sedang ada posisi → cari entry
        # ============================
        if not in_position:
            hour = current_time.hour
            # Time filter
            if hour < SESSION_START or hour > SESSION_END:
                continue

            # Wick filter: cek candle sebelumnya
            row_prev = df.iloc[i-1]
            if is_big_wick_candle(row_prev):
                continue

            # Hitung signal sampai bar ini saja
            window_df = df.iloc[: i+1].copy()
            signal_res, *_ = compute_signal_from_indicators(window_df)

            position = signal_res["position"]
            tp_p = signal_res["tp_price"]
            sl_p = signal_res["sl_price"]

            # HTF trend bias (4H)
            trend_bias = float(htf_on_ltf.iloc[i])
            if trend_bias == 1.0 and position == "SHORT":
                continue
            if trend_bias == -1.0 and position == "LONG":
                continue

            # Validasi TP/SL
            if position in ("LONG", "SHORT") and not (np.isnan(tp_p) or np.isnan(sl_p)):
                in_position = True
                pos_side = 1 if position == "LONG" else -1
                entry_price = current_close
                tp_price = tp_p
                sl_price = sl_p
                entry_time = current_time
                moved_to_be = False

                # Setup break-even rule parameter (hanya kalau trend kuat + TP cukup besar)
                adx_entry = signal_res.get("adx_last", np.nan)
                if pos_side == 1:  # LONG
                    full_tp_pct = (tp_price - entry_price) / entry_price
                else:              # SHORT
                    full_tp_pct = (entry_price - tp_price) / entry_price

                if (
                    not np.isnan(adx_entry)
                    and adx_entry >= 25          # trend kuat
                    and full_tp_pct is not None
                    and full_tp_pct >= 0.005   # min ~0.5% TP
                ):
                    be_allowed = True
                    be_full_tp_pct = full_tp_pct
                else:
                    be_allowed = False
                    be_full_tp_pct = None

                if verbose:
                    print(f"[{current_time}] OPEN {position} @ {entry_price:.2f}, "
                          f"TP={tp_price:.2f}, SL={sl_price:.2f}, "
                          f"Score={signal_res['score']}, Conf={signal_res['confidence_pct']:.1f}%, "
                          f"ADX={signal_res['adx_last']:.1f}, TrendBias={trend_bias}, "
                          f"Filter={signal_res['filter_reason']}")

    # Kalau di akhir masih ada posisi terbuka → close di close terakhir
    if in_position and entry_price is not None:
        final_close = float(close.iloc[-1])
        current_time = df.index[-1]

        if pos_side == 1:
            ret = (final_close - entry_price) / entry_price
        else:
            ret = (entry_price - final_close) / entry_price

        new_equity = equity[-1] * (1.0 + ret)
        equity.append(new_equity)
        equity_index.append(current_time)

        trades.append({
            "entry_time": entry_time,
            "exit_time": current_time,
            "side": "LONG" if pos_side == 1 else "SHORT",
            "entry_price": entry_price,
            "exit_price": final_close,
            "tp_price": tp_price,
            "sl_price": sl_price,
            "ret_pct": ret * 100.0,
            "exit_reason": "EOD_CLOSE",
            "adx_entry": adx_entry
        })

    if not trades:
        print("Tidak ada trade yang terjadi pada periode ini.")
        return None, None

    trades_df = pd.DataFrame(trades)
    equity_series = pd.Series(equity, index=equity_index)

    # ============================
    # Statistik backtest
    # ============================
    wins = trades_df[trades_df["ret_pct"] > 0]
    losses = trades_df[trades_df["ret_pct"] <= 0]

    total_trades = len(trades_df)
    winrate = len(wins) / total_trades * 100.0
    avg_win = wins["ret_pct"].mean() if not wins.empty else 0.0
    avg_loss = losses["ret_pct"].mean() if not losses.empty else 0.0
    total_return = (equity_series.iloc[-1] / equity_series.iloc[0] - 1.0) * 100.0

    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max
    max_dd = drawdown.min() * 100.0

    print("\n===== HASIL BACKTEST =====")
    print(f"Ticker         : {ticker}")
    print(f"Periode        : {start_date} s/d {end_date} ({interval})")
    print(f"Total trades   : {total_trades}")
    print(f"Winrate        : {winrate:.2f}%")
    print(f"Avg Win        : {avg_win:.2f}%")
    print(f"Avg Loss       : {avg_loss:.2f}%")
    print(f"Total Return   : {total_return:.2f}% (dari equity {initial_equity})")
    print(f"Max Drawdown   : {max_dd:.2f}%")

    plt.figure(figsize=(12, 5))
    plt.plot(equity_series.index, equity_series.values)
    plt.title(f"Equity Curve - {ticker}")
    plt.xlabel("Time")
    plt.ylabel("Equity (normalized)")
    plt.grid(True)
    plt.show()

    return trades_df, equity_series


# ============================
# Fungsi untuk widget (LIVE ANALISA)
# ============================
def interactive_plot(ticker, start_date, start_hour, dummy_forecast_period, interval):
    if start_date is None:
        print("⚠️ Silakan pilih Start Date terlebih dahulu.")
        return

    start_time = time(start_hour)
    # ✅ pastikan end time UTC aware (biar konsisten dan bisa dibandingkan dengan now_utc)
    start_datetime = datetime.combine(start_date, start_time).replace(tzinfo=timezone.utc)

    # Ambil 90 hari ke belakang dari start_datetime
    historical_start = start_datetime - timedelta(days=90)

    print(f"Mengambil data {ticker} dari {historical_start} sampai {start_datetime} (interval={interval}) ...")
    data = fetch_data(ticker, historical_start, start_datetime, interval)

    if data.empty:
        print("⚠️ Data kosong. Coba ubah tanggal / interval / ticker.")
        return

    df = data.dropna()

    # ✅ FIX: buang candle berjalan jika ini LIVE (kasus jam sama seperti sekarang)
    df = drop_incomplete_bar_if_live(df, interval, start_datetime)

    if df.empty:
        print("⚠️ Setelah buang candle LIVE, data jadi kosong.")
        return

    if len(df) < 60:
        print("⚠️ Data terlalu sedikit (butuh minimal ~60 bar).")
        return

    print("\n=== Sampel Data Close (5 bar terakhir) ===")
    print(df["Close"].tail())

    # Hitung signal
    signal_res, ema_fast, ema_slow, macd, macd_signal, rsi, adx = compute_signal_from_indicators(df)

    # ============================
    # Plot harga + EMA + MACD + RSI (TANPA ADX)
    # ============================
    close = df["Close"].astype(float)

    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(12, 10))

    # Chart 1: Harga + EMA
    axes[0].plot(close.index, close, label="Close")
    axes[0].plot(ema_fast.index, ema_fast, label="EMA 21")
    axes[0].plot(ema_slow.index, ema_slow, label="EMA 50")
    axes[0].set_title(f"{ticker} Price with EMA")
    axes[0].legend()
    axes[0].grid(True)

    # Chart 2: MACD
    axes[1].plot(macd.index, macd, label="MACD")
    axes[1].plot(macd_signal.index, macd_signal, label="Signal", linestyle="--")
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("MACD")
    axes[1].legend()
    axes[1].grid(True)

    # Chart 3: RSI
    axes[2].plot(rsi.index, rsi, label="RSI")
    axes[2].axhline(30, color="red", linestyle="--", linewidth=0.8)
    axes[2].axhline(70, color="green", linestyle="--", linewidth=0.8)
    axes[2].set_title("RSI")
    axes[2].legend()
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()

    # ============================
    # Ringkasan Trading Signal
    # ============================
    print("\n=== TRADING SIGNAL (EMA+MACD+RSI+ATR+ADX, BUKAN JAMINAN) ===")
    print(f"Raw Posisi (tanpa filter)          : {signal_res['raw_position']}")
    print(f"Posisi final (setelah semua filter): {signal_res['position']}")
    if signal_res['position'] != "NO TRADE":
        print(f"TP (saran) : {signal_res['tp_price']:.2f} ({signal_res['tp_pct']:.2f}%)")
        print(f"SL (saran) : {signal_res['sl_price']:.2f} ({signal_res['sl_pct']:.2f}%)")
    print(f"Confidence : {signal_res['confidence_pct']:.2f}%")
    print(f"Score      : {signal_res['score']} (semakin jauh dari 0, sinyal makin kuat)")
    print(f"Quality    : {signal_res['quality_label']}")
    print(f"ATR-based  : {signal_res['atr_based']} (True = TP/SL berdasarkan ATR)")
    print(f"ADX last   : {signal_res['adx_last']:.2f}")
    print(f"Last TR    : {signal_res['last_tr']:.4f}")
    print(f"Filter     : {signal_res['filter_reason']}")

    # Detail indikator
    print("\n=== DETAIL INDIKATOR TERAKHIR ===")
    for k, v in signal_res.items():
        print(f"{k:18s}: {v}")


# ============================
# WIDGET (LIVE ANALISA)
# ============================
ticker_widget = widgets.Text(value='ETH-USD', description='Ticker:')
start_date_widget = widgets.DatePicker(description='Start Date')
start_hour_widget = widgets.IntSlider(value=0, min=0, max=23, description='Start Hour:')
forecast_period_widget = widgets.IntText(value=1, description='(Ignored)')  # cuma biar UI mirip sebelumnya
interval_widget = widgets.Dropdown(
    options=[('1 Hour', '1h'), ('1 Day', '1d')],
    value='1h',
    description='Interval:'
)
refresh_button = widgets.Button(description="Analisa", button_style='info')

out = Output()

def on_button_clicked(b):
    with out:
        out.clear_output()
        start_date = start_date_widget.value
        start_hour = start_hour_widget.value
        interactive_plot(
            ticker_widget.value,
            start_date,
            start_hour,
            forecast_period_widget.value,  # tidak dipakai
            interval_widget.value
        )

refresh_button.on_click(on_button_clicked)

ui = VBox([
    ticker_widget,
    start_date_widget,
    start_hour_widget,
    interval_widget,
    forecast_period_widget,
    refresh_button,
    out
])

display(ui)
